# Lab 2: Entanglement and superdense coding

**Audience:** comfortable with Python and basic linear algebra; no prior quantum computing required for Lab 1.

## Learning objectives

- Read two-qubit states using Qiskit's bit-order convention.
- Prepare Bell states and distinguish entanglement from classical correlation.
- Measure correlations in two bases.
- Encode and recover two classical bits using a shared Bell pair and one transmitted qubit.

## How to use this notebook

Open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, and upload this file. Connect to a standard Python CPU runtime. Run cells in order with Shift+Enter. No IBM account, token, GPU, or paid service is needed. Internet access is needed for package installation only. If changing an already-used package version, restart the session after installation and rerun from the imports cell. These notebooks target Python 3.10–3.13.

For each experiment: **explain → predict → run → interpret → modify**. Write predictions and observations in a new text cell or your notes. Exercise starter cells are working examples, not finished answers to every requested change. Hints and solutions are collapsed below the prompts. Keep optional extensions until after the core lab.

## Agenda

| Activity |
|---|
| Recap, tensor products, bit ordering; Exercise 1 |
| CNOT and Bell states; Exercise 2 |
| Z/X correlations and classical mixture; Exercise 3 |
| Break and checkpoint |
| Superdense coding step by step |
| All four messages; Exercise 4 |
| 115–120 | Exit questions and resource accounting |

## Prerequisite recap

From Lab 1: X flips a bit, H creates or reverses superposition, and Z changes relative phase. Measurement probabilities are squared amplitude magnitudes. H immediately before measurement changes from the Z basis to the X basis. This notebook repeats all setup and helper code, so no earlier notebook needs to be open.

**Instructor guidance:** have pairs compare predictions before running code. A fast learner can attempt the optional extension while others finish.


## Setup

The first cell installs a fixed teaching environment. Installation may take a few minutes. The helper functions below build explicit classical measurement registers and use 2,048 shots with seed 42. A shot means one fresh preparation and measurement of the circuit. Fixed seeds make classroom results repeatable; they do not remove sampling uncertainty.

`exact` inspects a simulated state without measurement. `sample` returns finite-shot counts. `compare` plots both. Access to all amplitudes is a simulator convenience: a physical quantum computer does not hand you its state vector.


In [ ]:
%pip install -q qiskit==2.2.3 numpy==2.2.6 matplotlib==3.10.3 pylatexenc==2.10 sympy==1.14.0


In [ ]:
import sys
from importlib.metadata import version
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, DensityMatrix
from qiskit.primitives import StatevectorSampler

SEED, SHOTS = 42, 2048
print('Python:', sys.version.split()[0])
for package in ['qiskit', 'numpy', 'matplotlib', 'pylatexenc', 'sympy']:
    print(f'{package}: {version(package)}')

def exact(circuit):
    # Supply a circuit WITHOUT measurements. This is simulator-only inspection.
    return Statevector.from_instruction(circuit).probabilities_dict()

def sample(circuit, qubits=None):
    # Copy so that the original circuit stays available for state inspection.
    qubits = list(range(circuit.num_qubits)) if qubits is None else list(qubits)
    measured = circuit.copy()
    readout = ClassicalRegister(len(qubits), 'readout')
    measured.add_register(readout)
    measured.measure(qubits, readout)
    sampler = StatevectorSampler(seed=SEED)
    return sampler.run([measured], shots=SHOTS).result()[0].data.readout.get_counts()

def compare(circuit, title):
    probabilities, counts = exact(circuit), sample(circuit)
    labels = [format(i, f'0{circuit.num_qubits}b') for i in range(2**circuit.num_qubits)]
    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.bar(x - .18, [probabilities.get(k, 0) for k in labels], .36, label='Exact probability')
    ax.bar(x + .18, [counts.get(k, 0) / SHOTS for k in labels], .36, label=f'Sampled frequency ({SHOTS} shots)')
    ax.set(xticks=x, xticklabels=labels, ylim=(0, 1.12), ylabel='Probability / frequency', xlabel='Measured bit string', title=title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    print('Counts:', counts)
    return counts

def show(circuit):
    display(circuit.draw('mpl', fold=70))
    plt.show()


## Two qubits and bit ordering

Two qubits have four basis states. Qiskit displays strings as **q1 q0**, so `01` means q1=0 and q0=1. The amplitude array is ordered `00, 01, 10, 11`. The lowest-index qubit q0 is the least significant bit. Circuit diagrams still put q0 on the top wire.

For a product state, `|q₁⟩⊗|q₀⟩`, tensor products multiply the single-qubit amplitudes. For example, q1 in `|0⟩` and q0 in `|+⟩` gives `(|00⟩+|01⟩)/√2`. A two-qubit state need not be expressible as a product.


### Exercise 1 — Read the bit string

The baseline flips q0. Predict the only outcome. Change it to flip q1, then flip both. Finally prepare q1=0 and q0=plus and compare the vector with `np.kron`.

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

For Qiskit state vectors, write the tensor factors in descending qubit index.

</details>

<details><summary>Worked solution — open after attempting</summary>

X on q0 gives `01`; X on q1 gives `10`; both give `11`. H on q0 alone gives `[1,1,0,0]/sqrt(2)`, equal to `np.kron([1,0], [1,1]/sqrt(2))`.

</details>


In [ ]:
ordering = QuantumCircuit(2)
ordering.x(0)
show(ordering)
print('State:', Statevector.from_instruction(ordering).data)
print('Counts:', sample(ordering))
print('Example tensor product:', np.kron([1, 0], np.array([1, 1])/np.sqrt(2)))


## CNOT and Bell-state preparation

`cx(control, target)` flips the target if the control is 1 in each computational-basis component. The filled dot is the control and the circled plus is the target. CNOT maps basis states to basis states; on a superposition it acts linearly on each component.

Apply H to q0 and then CNOT from q0 to q1:

```text
|00⟩ → (|00⟩ + |01⟩)/√2 → (|00⟩ + |11⟩)/√2 = |Φ⁺⟩
```

This is a **Bell state**. It is entangled because it cannot be factored into two independent qubit states. Each qubit alone is random in Z measurements, while the pair always agrees. Correlation alone does not prove entanglement; we will compare another basis shortly.

**Predict:** Compare H on both qubits with a Bell pair. How many outcomes should each show?


In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
product = QuantumCircuit(2)
product.h([0, 1])
show(bell)
compare(bell, 'Bell pair: correlated in Z')
compare(product, 'Product of two plus states')
# A pure two-qubit state is a product exactly when its 2x2 coefficient matrix has rank 1.
for label, qc in [('Product', product), ('Bell', bell)]:
    coefficients = Statevector.from_instruction(qc).data.reshape(2, 2)
    print(label, 'coefficient rank:', np.linalg.matrix_rank(coefficients))


### Exercise 2 — Change a Bell state

Add X on q0 after Bell preparation. Predict which outcomes remain. Replace that X with Z on q0: inspect amplitudes as well as Z counts. Does unchanged Z measurement imply an unchanged state?

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

X swaps the q0 values; Z changes the sign of components with q0=1.

</details>

<details><summary>Worked solution — open after attempting</summary>

X gives `(|01⟩+|10⟩)/√2`, so the bits disagree. Z gives `(|00⟩-|11⟩)/√2`: the same Z probabilities as before but a different relative phase. X-basis correlations distinguish the two phase choices.

</details>


In [ ]:
changed_bell = bell.copy()
changed_bell.x(0)  # Replace with z(0) for the second experiment
print('Amplitudes:', np.round(Statevector.from_instruction(changed_bell).data, 3))
compare(changed_bell, 'Modified Bell pair')


## Correlation versus entanglement

Consider a classical source that chooses `00` or `11` with equal probability. This **mixture** has the same Z probabilities as `|Φ⁺⟩`. Its density matrix averages the outer products of those two states; the Bell state has additional off-diagonal terms representing coherence.

To measure both qubits in X, apply H to each before Z measurement. Bit 0 labels plus and bit 1 labels minus. The Bell pair still agrees in X. The classical mixture produces all four X outcomes equally often. This comparison distinguishes these two specified preparations. It is not a Bell-inequality experiment or a general entanglement certification protocol.

We sample the mixture by preparing each branch on half the shots and pooling counts. This fixed 50/50 allocation models the equal mixture without adding random variation to its branch weights.


In [ ]:
def in_basis(circuit, basis):
    rotated = circuit.copy()
    if basis == 'X': rotated.h([0, 1])
    return rotated

def mixture_experiment(basis):
    branches = []
    for bit in [0, 1]:
        qc = QuantumCircuit(2)
        if bit: qc.x([0, 1])
        branches.append(in_basis(qc, basis))
    rho = sum((DensityMatrix.from_instruction(qc).data for qc in branches)) / 2
    probabilities = DensityMatrix(rho).probabilities()
    counts = {label: 0 for label in ['00', '01', '10', '11']}
    for branch_index, qc in enumerate(branches):
        measured = qc.copy()
        readout = ClassicalRegister(2, 'readout')
        measured.add_register(readout)
        measured.measure([0, 1], readout)
        result = StatevectorSampler(seed=SEED + branch_index).run([measured], shots=SHOTS//2).result()[0]
        for key, count in result.data.readout.get_counts().items():
            counts[key] += count
    return probabilities, counts

def agreement(counts):
    return (counts.get('00', 0) + counts.get('11', 0)) / sum(counts.values())

for basis in ['Z', 'X']:
    compare(in_basis(bell, basis), f'Bell pair measured in {basis}')
    probabilities, counts = mixture_experiment(basis)
    print(basis, 'mixture exact probabilities:', probabilities)
    print(basis, 'mixture sampled counts:', counts, 'agreement:', agreement(counts))
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(list(counts), np.array(list(counts.values())) / SHOTS)
    ax.set(ylim=(0, 1), ylabel='Sampled frequency', xlabel='Measured bits', title=f'Classical mixture in {basis}')
    fig.tight_layout()
    plt.show()


### Exercise 3 — Detect a relative phase

Compare Phi-plus with the state obtained by Z on q0. Predict agreement in Z and X for each, then test. Explain why the Z data alone cannot distinguish them.

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

H converts plus/minus states to computational-basis outcomes.

</details>

<details><summary>Worked solution — open after attempting</summary>

Phi-plus agrees in both bases. Phi-minus agrees in Z but disagrees in X: after H on both, only `01` and `10` occur. In Z, squaring amplitudes loses the relative sign.

</details>


In [ ]:
phase_bell = bell.copy()
phase_bell.z(0)
for basis in ['Z', 'X']:
    counts = compare(in_basis(phase_bell, basis), f'Phi-minus in {basis}')
    print('Agreement:', agreement(counts))


## Break and checkpoint

Discuss: Why does “both bits agree” fail to identify an entangled source? Which additional experiment distinguished our Bell source from the specified classical source? Take a break.


## Superdense coding

**Task:** Alice wants Bob to recover a two-bit classical message `ab`. They already share a Bell pair. Alice holds q0 and Bob holds q1. Alice applies `Xᵇ` followed by `Zᵃ` on her qubit, then **sends q0 to Bob**. Bob now holds both qubits and decodes using CNOT(q0,q1), then H(q0).

We simulate the full process in one circuit; the barrier marks physical transmission, not a gate. Bob's final q0 equals a and q1 equals b. We explicitly measure q0 into classical bit 1 and q1 into classical bit 0, so the displayed classical message reads **ab**. This mapping differs from the default q1q0 state-vector label.

| Message ab | Alice's operations, in time order |
|---|---|
| 00 | Identity |
| 01 | X |
| 10 | Z |
| 11 | X then Z |

**Predict:** Choose a message and trace which Bell state Alice creates. What would Bob learn by measuring his original qubit before receiving Alice's?


In [ ]:
def dense_stages(message):
    if len(message) != 2 or set(message) - {'0', '1'}:
        raise ValueError('Use a two-bit string such as 01.')
    a, b = map(int, message)
    shared = QuantumCircuit(2)
    shared.h(0)
    shared.cx(0, 1)
    encoded = shared.copy()
    if b: encoded.x(0)
    if a: encoded.z(0)
    decoded = encoded.copy()
    decoded.barrier(label='Alice sends q0')
    decoded.cx(0, 1)
    decoded.h(0)
    return shared, encoded, decoded

def dense_counts(decoded):
    measured = decoded.copy()
    message_register = ClassicalRegister(2, 'message')
    measured.add_register(message_register)
    measured.measure(0, message_register[1])  # a, leftmost displayed bit
    measured.measure(1, message_register[0])  # b, rightmost displayed bit
    result = StatevectorSampler(seed=SEED).run([measured], shots=SHOTS).result()[0]
    show(measured)
    return result.data.message.get_counts()

message = '10'
shared, encoded, decoded = dense_stages(message)
for label, qc in [('Shared pair', shared), ('Encoded pair', encoded), ('Decoded q1q0', decoded)]:
    print(label, np.round(Statevector.from_instruction(qc).data, 3))
print('Recovered ab:', dense_counts(decoded))


### Interpret the resources

Two classical bits are recovered after Alice sends one qubit **and consumes one previously shared Bell pair**. Establishing that pair required earlier resources. Bob must receive Alice's qubit and perform joint decoding. His original qubit alone has a maximally mixed state, independent of the message; entanglement does not carry a controllable faster-than-light signal.

**Modify:** Try message `11`. Follow the phases through encoding, then explain why the decoder turns them into classical output bits.


### Exercise 4 — Recover all messages

Run all four messages and make a message→recovered-output table. Then remove the shared-pair preparation in a copy of the protocol and test message 10. Explain which resource was lost.

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

Loop over `00`, `01`, `10`, `11`. For the resource test, begin at `QuantumCircuit(2)` and apply the same encoding and decoding gates.

</details>

<details><summary>Worked solution — open after attempting</summary>

All four messages return themselves with 2,048 counts. Without the shared Bell pair, message 10 applies Z to zero (no observable change), then the decoder prepares a superposition on q0. The recovered messages are 00 and 10 with approximately equal frequency; deterministic recovery fails.

</details>


In [ ]:
recovered = {}
for message in ['00', '01', '10', '11']:
    _, _, decoded = dense_stages(message)
    # The helper displays the measurement wiring as well as returning counts.
    recovered[message] = dense_counts(decoded)
print(recovered)

without_pair = QuantumCircuit(2)
without_pair.z(0)  # Encode message 10 without initial entanglement
without_pair.cx(0, 1)
without_pair.h(0)
print('Without shared pair, recovered ab:', dense_counts(without_pair))


## Exit ticket

1. What does bit string `01` mean in the default Qiskit state ordering?
2. Why is a Bell pair different from independently prepared plus states?
3. What distinguishes our Bell state from the classical correlated mixture?
4. Which resources does superdense coding consume, and why must Alice send her qubit?

**Optional extension:** use `partial_trace` from `qiskit.quantum_info` to inspect Bob's qubit after each encoding. Show that all messages give the same reduced density matrix, identity/2.

## References

- [IBM: bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering)
- [IBM: entanglement in action](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action)
- [Qiskit DensityMatrix](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.DensityMatrix)
